In [20]:
import os
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import keras
import numpy as np
from huggingface_hub import login
from datasets import load_dataset
import seaborn as sns
import matplotlib.pyplot as plt
import dagshub
import mlflow
from dotenv import load_dotenv
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize
from tensorflow.keras.callbacks import EarlyStopping

In [21]:
# Load the dataset

df = pd.read_csv("datasets/Customer_Support_Tickets_Cleaned.csv")

In [22]:
# Only using English language tickets for BERT model training
#df = df[df['language'] == 'en']

In [23]:
df = df[['clean_sub_body', 'department']]

In [24]:
df

,clean_sub_body,department
0,wesentlicher sicherheitsvorfall sehr geehrtes ...,Technical
1,account disruption dear customer support team ...,Technical
2,query about smart home system integration feat...,Billing
3,inquiry regarding invoice details dear custome...,Billing
4,question about marketing agency software compa...,Sales
...,...,...
61758,assistance needed for ifttt docker integration...,Technical
61759,bitten um unterstützung bei der integration se...,Technical
61760,none hello customer support i am inquiring abo...,Billing
61761,hilfe bei digitalen strategie problemen die qu...,Technical


In [11]:


X = df['clean_sub_body']
le = LabelEncoder()

y = le.fit_transform(df['department'])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2, stratify=y)

In [12]:
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4", trainable=True)

In [13]:
# BERT

bert_tags = {
    "framework": "tensorflow.keras",
    "model_type": "BERT"
}

bert_train_params = {
    "optimizer": "adam",
    "loss": "sparse_categorical_crossentropy",
    "metrics": [
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")
    ],
    "epochs": 10,
    "batch_size": 32
}

bert_model_params = {
    "dropout": 0.1,
    "num_classes": len(le.classes_)
}

def create_bert_model(model_params):

    # Input Layer
    text_input = tf.keras.layers.Input(
        shape=(),
        dtype=tf.string,
        name="text"
    )

    # BERT preprocessing and encoder
    preprocessed_text = bert_preprocess(text_input)
    outputs = bert_encoder(preprocessed_text)

    # Classification Head
    x = tf.keras.layers.Dropout(
        model_params["dropout"],
        name="dropout"
    )(outputs["pooled_output"])

    output = tf.keras.layers.Dense(
        model_params["num_classes"],
        activation="softmax",
        name="output"
    )(x)

    model = tf.keras.Model(
        inputs=text_input,
        outputs=output
    )

    return model

bert_model = create_bert_model(bert_model_params)

In [14]:
models = [
    (
        "BERT",
        bert_model,
        (X_train, y_train),
        (X_test, y_test),
        bert_model_params,
        bert_train_params,
        bert_tags
    )
]

In [15]:
# ---------------------------------------------
# ROC Curve Function
# ---------------------------------------------

def create_roc_curve(model, model_name, X_test, y_test, labels):
    
    n_classes = len(labels)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)

    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)

    elif isinstance(model, tf.keras.Model):
        y_score = model.predict(X_test)

    else:
        raise ValueError(
            f"{type(model).__name__} does not support "
            "predict_proba or decision_function."
        )

    # ======================
    # Binary Classification
    # ======================
    if n_classes == 2:

        if y_score.ndim > 1:
            y_score_binary = y_score[:, 1]
        else:
            y_score_binary = y_score

        fpr, tpr, _ = roc_curve(
            y_test,
            y_score_binary
        )

        roc_auc = auc(fpr, tpr)

        fig, ax = plt.subplots(figsize=(7, 5))

        ax.plot(
            fpr,
            tpr,
            lw=2,
            label=f"AUC = {roc_auc:.3f}"
        )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Random Classifier"
        )

    # ======================
    # Multiclass Classification
    # ======================
    else:

        # One-hot encode y_test
        y_test_bin = label_binarize(
            y_test,
            classes=np.arange(n_classes)
        )

        roc_auc = roc_auc_score(
            y_test_bin,
            y_score,
            multi_class="ovr",
            average="weighted"
        )

        fig, ax = plt.subplots(figsize=(8, 6))

        for i in range(n_classes):

            fpr, tpr, _ = roc_curve(
                y_test_bin[:, i],
                y_score[:, i]
            )

            class_auc = auc(fpr, tpr)

            ax.plot(
                fpr,
                tpr,
                lw=1.5,
                label=f"{labels[i]} (AUC={class_auc:.3f})"
            )

        ax.plot(
            [0, 1],
            [0, 1],
            linestyle="--",
            label="Random Classifier"
        )

    # ======================
    # Common Plot Formatting
    # ======================
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve — {model_name}")

    ax.legend(
        loc="lower right",
        fontsize=8
    )

    plt.tight_layout()

    # Save plot
    save_dir = os.path.join(
        "Images",
        "ROC_Curves"
    )

    os.makedirs(
        save_dir,
        exist_ok=True
    )

    save_path = os.path.join(
        save_dir,
        f"{model_name}.png"
    )

    fig.savefig(
        save_path,
        dpi=150
    )

    plt.close(fig)

    print(
        f"ROC Curve saved → {save_path} "
        f"(AUC: {roc_auc:.3f})"
    )

    return save_path, roc_auc

# ---------------------------------------------
# Confusion Matrix Function
# ---------------------------------------------

def create_confusion_matrix(model_name, y_true, y_pred, class_names, path="Images"):
    
    cm = confusion_matrix(y_true, y_pred)

    save_dir = os.path.join(path, "Confusion Matrix")
    os.makedirs(save_dir, exist_ok=True)

    safe_name = model_name.replace(" ", "_")
    save_path = os.path.join(save_dir, f"{safe_name}.png")

    plt.figure(figsize=(8, 8))

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )

    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

    print(f"Confusion Matrix saved → {save_path}")

    return save_path

In [16]:
model_reports = []
model_history = []
model_roc_paths = []
model_roc_auc_scores = []
model_cm_path = []

for model_name, model, train_set, test_set, model_params, train_params, tags in models:

    X_train, Y_train = train_set
    X_test, Y_test = test_set
    

    # Model Training
    model.compile(
        optimizer=train_params['optimizer'],
        loss=train_params['loss'],
        metrics=train_params['metrics']
    )


    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        min_delta=0.001
    )


    history = model.fit(
        X_train,
        Y_train,
        epochs=train_params['epochs'],
        batch_size=train_params['batch_size'],
        callbacks=[early_stop]
    )
    model_history.append(history)


    # Model Evaluation
    predictions = model.predict(X_test)
    y_pred = np.argmax(predictions, axis=1)


    # Saving Keras Model Locally
    model_save_dir = "saved_models"
    os.makedirs(model_save_dir, exist_ok=True)
    model_save_path = os.path.join(model_save_dir, f"{model_name}.keras")
    model.save(model_save_path)

    print(f"{model_name} saved to {model_save_path}")


    # Classification Report
    labels = le.classes_
    report = classification_report(
        Y_test,
        y_pred,
        target_names=labels,
        output_dict= True
    )
    model_reports.append(report)


    # ROC Curve
    roc_path, roc_auc = create_roc_curve(model, model_name, X_test, Y_test, labels)
    model_roc_paths.append(roc_path)
    model_roc_auc_scores.append(roc_auc)
    print(f"{model_name} ROC AUC saved successfully!")


    # Confusion Matrix
    cm_path = create_confusion_matrix(
        model_name=model_name,
        y_true=Y_test,
        y_pred=y_pred,
        class_names=le.classes_
    )
    model_cm_path.append(cm_path)
    print(f"{model_name} Confusion Matrix saved successfully!")
    print("--------------------------------------------------------")

Epoch 1/10
707/707 [==============================] - ETA: 0s - loss: 1.1307 - accuracy: 0.6263WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 386s 529ms/step - loss: 1.1307 - accuracy: 0.6263
Epoch 2/10
707/707 [==============================] - ETA: 0s - loss: 1.0961 - accuracy: 0.6330WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 375s 530ms/step - loss: 1.0961 - accuracy: 0.6330
Epoch 3/10
707/707 [==============================] - ETA: 0s - loss: 1.0874 - accuracy: 0.6331WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 373s 527ms/step - loss: 1.0874 - accuracy: 0.6331
Epoch 4/10
707/707 [==============================] - ETA: 0s - loss: 1.0854 - accuracy: 0.6331WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 373s 528ms/step - loss: 1.0854 - accuracy: 0.6331
Epoch 5/10
707/707 [==============================] - ETA: 0s - loss: 1.0846 - accuracy: 0.6331WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


...

707/707 [==============================] - 370s 524ms/step - loss: 1.0823 - accuracy: 0.6331
Epoch 9/10
707/707 [==============================] - ETA: 0s - loss: 1.1576 - accuracy: 0.6234WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


707/707 [==============================] - 371s 525ms/step - loss: 1.1576 - accuracy: 0.6234
Epoch 10/10
707/707 [==============================] - ETA: 0s - loss: 1.1259 - accuracy: 0.6327WARNING:tensorflow:Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,accuracy


177/177 [==============================] - 41s 229ms/step
BERT saved to saved_models\BERT.keras
  1/177 [..............................] - ETA: 21s

c:\Users\jstha\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\jstha\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\jstha\miniconda3\envs\tf\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


177/177 [==============================] - 41s 232ms/step
ROC Curve saved → Images\ROC_Curves\BERT.png (AUC: 0.506)
BERT ROC AUC saved successfully!
Confusion Matrix saved → Images\Confusion Matrix\BERT.png
BERT Confusion Matrix saved successfully!
--------------------------------------------------------


In [18]:
dagshub.init(repo_owner='JS-Tharun', repo_name='Customer-Support-Automation', mlflow=True)

load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('MLFLOW_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('MLFLOW_PASSWORD')}"

mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    X_train, Y_train = element[2]
    X_test, Y_test = element[3]
    model_params = element[4]
    train_params = element[5]
    tags = element[6]

    model_report = model_reports[i]
    history = model_history[i]
    model_roc_auc = model_roc_auc_scores[i]
    roc_path = model_roc_paths[i]

    with mlflow.start_run(run_name=model_name):

        # ---------------------------------------------------
        # Logging Parameters and Tags
        # ---------------------------------------------------

        mlflow.set_tags(tags)

        mlflow.log_params(model_params)
        mlflow.log_params(train_params)

        print(f"{model_name} parameters and tags logged!")

        #-----------------------------------------------------
        # Model training and validation performance tracking
        #-----------------------------------------------------

        for epoch in range(len(history.history['loss'])):
            mlflow.log_metric("train_loss", history.history['loss'][epoch], step=epoch)
            mlflow.log_metric("train_accuracy", history.history['accuracy'][epoch], step=epoch)

        print(f"{model_name} training and validation performance logged!")

        #------------------------------------------------------
        # Log Model
        #------------------------------------------------------

        model_path = os.path.join("saved_models", f"{model_name}.keras")

        if os.path.exists(model_path):
            keras_model = keras.models.load_model(
                model_path,
                custom_objects={'KerasLayer': hub.KerasLayer}
            )
            model_info = mlflow.keras.log_model(
                keras_model,
                artifact_path=model_name
            )
            print(f"{model_name} model logged!")

        else:
            print(f"{model_name} model file not found at {model_path}!")


        #------------------------------------------------------
        # Model testing performance tracking
        #------------------------------------------------------

        # Log best epoch metrics explicitly (for clean experiment table comparison)
        best_epoch = np.argmin(history.history['loss'])
        mlflow.log_metric("best_epoch", best_epoch)
        mlflow.log_metric("best_train_accuracy", history.history['accuracy'][best_epoch])

        # Link test metrics to the LoggedModel using model_id
        mlflow_metrics = {}
        for label, metrics in model_report.items():
            if isinstance(metrics, dict):
                for metric_name, value in metrics.items():
                    if metric_name != "support":
                        mlflow_metrics[f"{label}_{metric_name}"] = float(value)
            else:
                mlflow_metrics[label] = float(metrics)

        # Pass model_id to link metrics to the model in the Models tab
        mlflow.log_metrics(mlflow_metrics, model_id=model_info.model_id)

        print(f"{model_name} metrics logged")

        # Log ROC AUC metric
        mlflow_metrics["roc_auc"] = float(model_roc_auc)   # ← AUC as a metric
        mlflow.log_metrics(mlflow_metrics)
        print("Metrics Logged")

        # Log ROC AUC curve image
        mlflow.log_artifact(roc_path, artifact_path="plots/ROC AUC Curve")

        # Log Confusion matrix image
        mlflow.log_artifact(model_cm_path[i], artifact_path="plots/Confusion Matrix")

Initialized MLflow to track repo "JS-Tharun/Customer-Support-Automation"

Repository JS-Tharun/Customer-Support-Automation initialized!

BERT parameters and tags logged!
BERT training and validation performance logged!


2026/06/03 15:34:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/03 15:34:16 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmplhb7k577\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmplhb7k577\model\data\model\assets
2026/06/03 15:34:45 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\jstha\AppData\Local\Temp\tmplhb7k577\model, flavor: tensorflow). Fall back to return ['tensorflow==2.10.1']. Set logging level to DEBUG to see the full traceback. 
2026/06/03 15:34:45 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


BERT model logged!
BERT metrics logged
Metrics Logged
🏃 View run BERT at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0/runs/9cd165ea087e4f1680883482a574e965
🧪 View experiment at: https://dagshub.com/JS-Tharun/Customer-Support-Automation.mlflow/#/experiments/0


In [ ]:
"""# BERT layers
text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessed_text = bert_preprocess(text_input)
outputs = bert_encoder(preprocessed_text)

# Classification head
l = tf.keras.layers.Dropout(
    0.1,
    name="dropout"
)(outputs['pooled_output'])

l = tf.keras.layers.Dense(
    len(le.classes_),
    activation='softmax',
    name="output"
)(l)

# Model
bert_model = tf.keras.Model(
    inputs=text_input,
    outputs=l
)

METRICS = [
    tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')
]

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=METRICS
)

model.fit(X_train, y_train, epochs=10)"""

Epoch 1/10
1545/1545 [==============================] - 372s 236ms/step - loss: 1.1777 - accuracy: 0.5978
Epoch 2/10
1545/1545 [==============================] - 381s 246ms/step - loss: 1.1094 - accuracy: 0.6228
Epoch 3/10
1545/1545 [==============================] - 364s 236ms/step - loss: 1.0907 - accuracy: 0.6281
Epoch 4/10
1545/1545 [==============================] - 369s 239ms/step - loss: 1.0826 - accuracy: 0.6320
Epoch 5/10
1545/1545 [==============================] - 362s 234ms/step - loss: 1.0775 - accuracy: 0.6340
Epoch 6/10
1545/1545 [==============================] - 363s 235ms/step - loss: 1.0698 - accuracy: 0.6388
Epoch 7/10
1545/1545 [==============================] - 361s 234ms/step - loss: 1.0674 - accuracy: 0.6381
Epoch 8/10
1545/1545 [==============================] - 363s 235ms/step - loss: 1.0647 - accuracy: 0.6385
Epoch 9/10
1545/1545 [==============================] - 365s 236ms/step - loss: 1.0630 - accuracy: 0.6400
Epoch 10/10
1545/1545 [=======================